# Room3D GPU server (Kaggle)

A **reusable server notebook**: run all cells once, copy the printed `https://….trycloudflare.com` URL and token into the website
(*GPU server* badge). The website sends requests (upload → job → poll → download); this notebook only runs the FastAPI server in
`gpu_server/` and keeps it alive. Nothing here is specific to one project.

**Settings → Accelerator: GPU T4 ×2**, **Internet: on**. Add Kaggle *Secrets* (Add-ons → Secrets): `GPU_SERVER_TOKEN` (any long random string) and `HF_TOKEN` (Hugging Face token; needed for SAM 3 / gated FLUX files).

GPU split: **cuda:0** = FLUX.2 klein transformer + text encoder + VAE, **cuda:1** = SAM + 3DGS (gsplat). With one GPU everything shares cuda:0 and models are swapped in and out (`POST /models/unload`).


In [ ]:
import os, subprocess, sys, time, json, secrets, pathlib
REPO_URL = "https://github.com/just0neball/room-3d-reconstruction.git"
BRANCH   = "main"            # branch that contains gpu_server/
MOCK     = False             # True = synthetic scene + CPU stand-ins (no models, no GPU): only for trying the website
PORT     = 8001

# model / memory options (all optional; see gpu_server/settings.py)
os.environ.setdefault("SAM_MODEL_ID", "facebook/sam2.1-hiera-small")      # or "facebook/sam3" (gated)
os.environ.setdefault("DIFFUSION_MODEL_ID", "black-forest-labs/FLUX.2-klein-4B")
os.environ.setdefault("DIFFUSION_QUANT", "nf4")                           # nf4 | int8 | bf16 (+cpu offload)
os.environ.setdefault("EDIT_LONG_SIDE", "768")                            # crop sent to the diffusion model (multiple of 16)
os.environ["GPU_MOCK"] = "1" if MOCK else "0"

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for name in ("GPU_SERVER_TOKEN", "HF_TOKEN"):
        try: os.environ[name] = _s.get_secret(name)
        except Exception: pass
except ImportError:
    pass
if not os.environ.get("GPU_SERVER_TOKEN"):
    os.environ["GPU_SERVER_TOKEN"] = secrets.token_urlsafe(24)
    print("No GPU_SERVER_TOKEN secret found - generated one for this session.")
TOKEN = os.environ["GPU_SERVER_TOKEN"]
if os.environ.get("HF_TOKEN"):
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]


In [ ]:
WORK = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle").exists() else pathlib.Path.cwd()
REPO = WORK / "room-3d-reconstruction"
if REPO.exists():
    !git -C {REPO} fetch origin {BRANCH} && git -C {REPO} checkout {BRANCH} && git -C {REPO} pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO}
os.chdir(REPO); sys.path.insert(0, str(REPO))
!pip install -q --break-system-packages -r gpu_server/requirements.txt 2>&1 | tail -3


In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,memory.used --format=csv
if not MOCK:
    import torch; print("torch", torch.__version__, "| cuda devices:", torch.cuda.device_count())
    import diffusers, transformers; print("diffusers", diffusers.__version__, "| transformers", transformers.__version__)


## Start the server

In [ ]:
LOG = open(WORK / "gpu_server.log", "w")
env = {**os.environ, "PYTHONPATH": str(REPO), "PYTHONUNBUFFERED": "1"}
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "gpu_server.app:app", "--host", "0.0.0.0", "--port", str(PORT), "--log-level", "info"],
                          cwd=REPO, env=env, stdout=LOG, stderr=subprocess.STDOUT)
import requests
H = {"X-Room3D-Token": TOKEN}
r = None
for _ in range(60):
    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/health", headers=H, timeout=3)
        if r.ok: break
    except Exception: time.sleep(1)
print(json.dumps(r.json(), indent=1) if r is not None else "server did not start - see the log cell below")


## Public URL (Cloudflare quick tunnel)

In [ ]:
if not (WORK / "cloudflared").exists():
    !curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o {WORK}/cloudflared && chmod +x {WORK}/cloudflared
import re, threading
tun = subprocess.Popen([str(WORK / "cloudflared"), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None; t0 = time.time()
while time.time() - t0 < 60 and not URL:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tun.stdout.readline())
    if m: URL = m.group(0)
print("\nGPU server URL :", URL, "\nToken          :", TOKEN)
print("Website -> GPU badge -> paste both. (The URL changes every session.)")
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()   # keep draining so the tunnel never blocks


In [ ]:
# check from the outside, exactly like the website does
r = requests.get(f"{URL}/health", headers=H, timeout=20); print(r.status_code, json.dumps(r.json(), indent=1))


## Smoke tests (optional, real models)
Each cell loads one model on its GPU and prints peak VRAM, so the install can be confirmed before the demo.

In [ ]:
# SAM: segment a click on a synthetic image
if not MOCK:
    import numpy as np, torch
    from gpu_server import models, settings
    seg = models.segmenter(); dev = settings.devices()[1]
    img = np.zeros((480, 640, 3), np.uint8); img[150:350, 200:420] = (200, 80, 60)
    t0 = time.time(); mask, score = seg.segment(img, [(300, 250, 1)])
    print(type(seg).__name__, "| mask px:", int(mask.sum()), "(expected ~44000) | score %.2f | %.1fs" % (score, time.time() - t0))
    print("peak VRAM on", dev, round(torch.cuda.max_memory_allocated(dev) / 1024**3, 2), "GB")


In [ ]:
# FLUX.2 klein: masked-crop edits per quantisation -> docs/reports/week2_benchmark.csv (load s, sec/frame, peak VRAM)
FRAME = ""   # path to any JPG, e.g. a frame the website already uploaded
if FRAME and not MOCK:
    !python -m gpu_server.editing.benchmark --image {FRAME} --quants nf4 int8 bf16 --frames 6
    import pandas as pd; display(pd.read_csv("docs/reports/week2_benchmark.csv"))


## While it runs
* Logs: run the cell below · per-stage timing and VRAM of every job: `gpu_data/metrics.csv`.
* Free GPU memory between jobs: `requests.post(f"{URL}/models/unload", headers=H)`.
* Kaggle ends idle / long sessions: re-run the notebook and paste the new URL; the website re-uploads anything the server forgot.


In [ ]:
!tail -n 25 {WORK}/gpu_server.log


In [ ]:
# stop everything
tun.terminate(); server.terminate()
